![Pythonista](img/pythonista.png)

# *Terraform*: fundamentos y estado remoto.

Esta sesión establece la base operativa de *IaC* para el resto del bloque: *GitHub Actions* como plano de control, test en *Linux* con fines didácticos y prod en *GCP* como destino gestionado.

## Objetivos.

- Entender `provider`, variables, `outputs` y módulos.
- Introducir *backend* remoto y locking para trabajo colaborativo.
- Preparar la integración con *GitHub Actions*, *OIDC* y aprobaciones por entorno.

En el codebase actual, la base de *Terraform* vive en `infra/terraform-py271` y está parametrizada para consumir configuración desde variables y secretos por entorno en *GitHub Actions*.

## *IaC*, *DevSecOps* y el rol del desarrollador.

### Cómo se conecta *IaC* con *DevSecOps*.

*DevSecOps* no es una herramienta sino un principio: integrar seguridad en cada fase del ciclo, desde el diseño hasta la operación, sin delegarla a un equipo separado al final del proceso. *IaC* es el vínculo operativo que hace posible ese principio en la capa de infraestructura.

| Principio *DevSecOps* | Cómo lo materializa *IaC* |
|---|---|
| *Shift-left* de seguridad | Los controles (firewall, IAM, cifrado) se declaran en el mismo *commit* que abre el recurso, no se aplican a posteriori. |
| Revisión entre pares | Los cambios de infraestructura pasan por *pull request* y pueden ser revisados por el equipo igual que el código. |
| Trazabilidad y auditoría | `git log` muestra quién aprobó qué cambio y cuándo; el artefacto del `plan` queda almacenado junto a la evidencia de ejecución. |
| Reproducibilidad de entornos | Dev, test y prod se crean desde las mismas plantillas, eliminando diferencias de configuración que ocultan vulnerabilidades. |
| Mínimo privilegio por diseño | Los roles *IAM*, reglas de red y permisos de servicio se definen explícitamente y se revisan en *PR*, no se acumulan por conveniencia. |
| Detección de deriva | `terraform plan` muestra inmediatamente si alguien modificó infraestructura fuera del *pipeline*, restaurando el estado declarado. |

En la práctica, un *pipeline* *DevSecOps* con *IaC* luce así:

```
PR con cambio .tf
  → lint + validate (quality gate)
  → terraform plan (visibilidad de impacto)
  → revisión humana + aprobación de entorno
  → terraform apply (ejecución controlada)
  → smoke tests + validación de seguridad
  → evidencia de auditoría archivada
```

Cada etapa es un control. Si una falla, el cambio no avanza. La seguridad no es un paso adicional al final: está distribuida en toda la cadena.

### Por qué un desarrollador necesita nociones de *IaC*.

Un desarrollador moderno no opera en un vacío: su código corre sobre infraestructura que alguien tiene que describir, versionar y mantener. Cuanto más alejado esté del proceso, más fricciones acumula:

- El entorno de prod lo sorprende: sin conocer la configuración de red o los permisos de la función que despliega, el primer error en producción parece magia negra;
- Rompe contratos sin saberlo: un cambio de puerto, variable de entorno o rol *IAM* que parece trivial puede invalidar reglas de seguridad declaradas en *Terraform*;
- No puede contribuir a la cultura *DevSecOps*: si la seguridad de la infraestructura queda confinada a un especialista, el equipo vuelve al modelo de silo que *DevSecOps* busca eliminar;
- El *pipeline* le habla en *IaC*: los mensajes de `plan`, los errores de `validate` y los bloqueos de *policy* son parte del flujo de trabajo diario;
- La arquitectura de la aplicación dicta la infraestructura: decidir entre *Cloud Run*, *Cloud Run functions* o *Compute Engine* afecta latencia, costos, escalado y superficie de ataque.

El objetivo no es convertir al desarrollador en *SRE*. Es que pueda leer, entender e influir en la infraestructura de su aplicación, y que reconozca cuándo un cambio que parece de código tiene consecuencias de infraestructura.

## Introducción desde cero: qué es *IaC* y por qué *Terraform*.

Si vienes de desarrollo de software y no has trabajado con infraestructura como código (*IaC*), piensa en [*Terraform*](https://developer.hashicorp.com/terraform) como el equivalente de declarar el estado deseado de tu entorno y versionarlo en *Git*.

Con *IaC* pasamos de cambios manuales en consola a cambios revisables y repetibles:

- antes: crear recursos a mano (sin trazabilidad completa);
- ahora: declarar recursos en archivos `.tf`, revisarlos por pull request y aplicar cambios de forma controlada.

## Vocabulario mínimo.

- `provider`: plugin que sabe hablar con una nube (por ejemplo *GCP*).
- `resource`: objeto real a crear o modificar (bucket, red, VM, función).
- `variable`: dato de entrada para parametrizar configuraciones.
- `output`: dato de salida útil para otros módulos o *pipelines*.
- `state`: registro de lo que *Terraform* ya administra.
- `backend remoto`: ubicación compartida del *state* para trabajar en equipo.
- `HCL` (*HashiCorp Configuration Language*): lenguaje declarativo legible por humanos para describir infraestructura.

Ejemplo mínimo de definición en `HCL`:

```hcl
provider "google" {
  project = var.project_id
  region  = var.region
}

resource "google_storage_bucket" "tf_state" {
  name     = var.tf_state_bucket
  location = var.region
}
```

## Ciclo de trabajo recomendado (operación normal).

1. Preparación del entorno: `terraform init` y selección de variables por entorno (`dev`, `staging`, `prod`, o el modelo que adopte tu organización).
2. Higiene del código: `terraform fmt` y `terraform validate` para evitar deuda técnica y errores de sintaxis.
3. Lectura de impacto: `terraform plan -out=tfplan` para revisar cambios esperados antes de ejecutar.
4. Revisión y aprobación: compartir el plan en el flujo de revisión (PR + aprobación de entorno).
5. Ejecución controlada: `terraform apply tfplan` para aplicar exactamente lo aprobado.
6. Verificación posterior: comprobar outputs, salud del servicio y controles de seguridad.
7. Trazabilidad: conservar plan, commit y evidencia de ejecución en CI/CD.

### Controles de endurecimiento dentro del ciclo.

- identidad: usar OIDC y cuentas de servicio separadas por rol (deployer vs runtime), sin llaves estáticas;
- mínimo privilegio: limitar IAM por recurso, entorno y acción;
- superficie de red: denegar por defecto, abrir puertos mínimos y preferir accesos controlados (por ejemplo, bastión/IAP);
- secretos: no exponer credenciales en código, variables de CI ni `terraform.tfstate`;
- gobernanza: bloquear `apply` directo en ramas no autorizadas y exigir revisión humana para entornos sensibles;
- trazabilidad: conservar artefactos de `plan` y evidencia de aprobación para auditoría.

Comandos de referencia:

```bash
terraform init
terraform fmt -check
terraform validate
terraform plan -out=tfplan
terraform apply tfplan
```

Documentación oficial por comando:

- [`terraform init`](https://developer.hashicorp.com/terraform/cli/commands/init)
- [`terraform fmt`](https://developer.hashicorp.com/terraform/cli/commands/fmt)
- [`terraform validate`](https://developer.hashicorp.com/terraform/cli/commands/validate)
- [`terraform plan`](https://developer.hashicorp.com/terraform/cli/commands/plan)
- [`terraform apply`](https://developer.hashicorp.com/terraform/cli/commands/apply)

## Cuando hay que rehacer infraestructura.

No siempre conviene "parchar". Rehacer puede ser razonable cuando:

- existe deriva grande entre estado real y código;
- hay cambios estructurales (redes, identidad, segmentación) difíciles de migrar en caliente;
- un incidente obliga a reconstrucción limpia con controles reforzados.

Flujo recomendado para reconstrucción segura:

1. Congelar cambios: pausar `apply` automáticos y trabajar en rama dedicada.
2. Respaldar estado: asegurar copia del *state* remoto y bloquear concurrencia.
3. Definir estrategia:
   - recreación completa: destruir y recrear;
   - recreación parcial: reemplazar solo módulos/recursos críticos (`-replace` o cambios por módulo).
4. Simular primero: `terraform plan -destroy` (si aplica) o `terraform plan` de reemplazo para dimensionar impacto.
5. Ejecutar por etapas: priorizar dependencias base (red, IAM, secretos) antes de cómputo y despliegue.
6. Rehidratar integraciones: DNS, permisos, monitoreo, alertas y pipelines.
7. Validar recuperación: smoke tests, revisión de seguridad y evidencia de compliance.

### Endurecimiento específico tras una reconstrucción.

- rotar secretos, tokens y credenciales de servicio aunque no se sospeche compromiso;
- reaplicar baseline de hardening de SO e imagen (usuarios, SSH, paquetes mínimos, configuración segura);
- verificar políticas organizacionales, etiquetas y controles de cumplimiento;
- reevaluar reglas de firewall y rutas para evitar aperturas temporales que queden permanentes;
- ejecutar pruebas de seguridad y operabilidad antes de declarar servicio restaurado.

Comandos útiles en escenarios de rehacer:

```bash
# Inspección y respaldo lógico
terraform state list
terraform show

# Simulación de destrucción total (solo para revisión)
terraform plan -destroy

# Reemplazo controlado de un recurso
terraform apply -replace="google_compute_instance.web"

# Destrucción explícita (solo con aprobación formal)
terraform destroy
```

Documentación oficial por comando:

- [`terraform state list`](https://developer.hashicorp.com/terraform/cli/commands/state/list)
- [`terraform show`](https://developer.hashicorp.com/terraform/cli/commands/show)
- [`terraform plan -destroy`](https://developer.hashicorp.com/terraform/cli/commands/plan#planning-modes)
- [`terraform apply -replace`](https://developer.hashicorp.com/terraform/cli/commands/apply#replace-address)
- [`terraform destroy`](https://developer.hashicorp.com/terraform/cli/commands/destroy)

La idea central: *Terraform* no "adivina". Compara declaración + *state* y produce un *plan* explícito antes de tocar infraestructura. En reconstrucciones, esa disciplina evita pérdidas y reduce riesgo operacional.

## Alternativas a *Terraform* (con referencias).

No existe una única herramienta correcta para todos los equipos. Elegir depende del ecosistema cloud, nivel de control requerido y madurez operativa.

| Herramienta | Enfoque | Cuándo suele encajar | Referencia |
|---|---|---|---|
| **Pulumi** | IaC con lenguajes de propósito general (Python, TypeScript, Go, etc.) | Equipos que prefieren reutilizar librerías y patrones de software en IaC | https://www.pulumi.com/docs/iac/ |
| **AWS CloudFormation** | Plantillas nativas de AWS | Equipos 100% AWS que priorizan integración nativa del proveedor | https://docs.aws.amazon.com/cloudformation/ |
| **Azure Bicep** | DSL declarativo para Azure | Equipos centrados en Azure con sintaxis más simple que ARM JSON | https://learn.microsoft.com/azure/azure-resource-manager/bicep/ |
| **Google Cloud Deployment Manager** | IaC nativo histórico en GCP | Contextos legados en GCP (evaluar alternativas actuales en proyectos nuevos) | https://cloud.google.com/deployment-manager |
| **Crossplane** | Control plane en Kubernetes para aprovisionar cloud | Organizaciones que estandarizan operaciones sobre Kubernetes | https://docs.crossplane.io/ |

Para el bloque actual se usa *Terraform* porque es multi-cloud, ampliamente adoptado y facilita conectar `plan/apply` con gobierno por entorno en *pipelines* de *CI/CD*.

## *Terraform* dentro del *pipeline* y estrategia de *runners*.

Los comandos `terraform fmt`, `terraform validate` y `terraform plan` son equivalentes de infraestructura a los *quality gates* vistos antes. Antes de aplicar cambios, el *pipeline* debe validar sintaxis, consistencia y *plan* resultante.

En la mayoría de casos:

- `plan` corre en hosted *runners*;
- `apply` puede correr en hosted si no requiere red privada;
- self-hosted se reserva para necesidades técnicas justificadas (acceso interno, segmentación o cumplimiento).

El estado remoto importa porque varios operadores o *workflows* pueden intentar trabajar sobre la misma infraestructura. Sin *backend* compartido y locking, el riesgo operacional crece de inmediato.

## Puente operativo del bloque.

Este cuaderno prepara la base para:

- definir el destino productivo en *GCP* con *Terraform*;
- gobernar `plan` y `apply` en el *pipeline*;
- mantener despliegue *Linux* de test bajo los mismos controles de trazabilidad.

## Mini actividad: decidir IaC con prioridad de endurecimiento.

Instrucciones: para cada escenario, selecciona una herramienta y justifica la decisión con enfoque de seguridad primero.

Criterios obligatorios (en este orden):

1. endurecimiento e identidad (OIDC, mínimo privilegio, separación de roles);
2. gobernanza y trazabilidad (aprobaciones, auditoría, reproducibilidad);
3. operabilidad (mantenibilidad, curva de aprendizaje, costo de cambio).

Escenarios:

1. Equipo multi-cloud con aprobaciones por entorno y auditoría estricta.
2. Equipo 100% AWS que necesita integración nativa, pero sin usar llaves estáticas en CI/CD.
3. Equipo 100% Azure que busca simplicidad declarativa y cumplimiento corporativo.
4. Incidente de seguridad obliga a rehacer infraestructura crítica con ventana limitada y validación posterior obligatoria.

Salida esperada por escenario:

- herramienta propuesta;
- justificación breve (3 a 5 líneas) priorizando controles de endurecimiento;
- riesgo principal y mitigación concreta;
- criterio de salida: qué evidencia usarías para aprobar el cambio en pipeline.

In [ ]:
# Autoevaluación orientada a endurecimiento.
# El puntaje combina: seguridad (50%), gobernanza (30%), operabilidad (20%).

pesos = {
    "seguridad": 0.5,
    "gobernanza": 0.3,
    "operabilidad": 0.2,
}

escenarios = {
    "multi_cloud_auditoria_estricta": {
        "terraform": {"seguridad": 5, "gobernanza": 5, "operabilidad": 4},
        "pulumi": {"seguridad": 4, "gobernanza": 4, "operabilidad": 4},
        "cloudformation": {"seguridad": 3, "gobernanza": 3, "operabilidad": 2},
        "bicep": {"seguridad": 3, "gobernanza": 3, "operabilidad": 2},
    },
    "aws_sin_llaves_estaticas": {
        "terraform": {"seguridad": 5, "gobernanza": 4, "operabilidad": 4},
        "pulumi": {"seguridad": 4, "gobernanza": 3, "operabilidad": 4},
        "cloudformation": {"seguridad": 5, "gobernanza": 4, "operabilidad": 4},
        "bicep": {"seguridad": 1, "gobernanza": 1, "operabilidad": 1},
    },
    "azure_cumplimiento_corporativo": {
        "terraform": {"seguridad": 4, "gobernanza": 4, "operabilidad": 4},
        "pulumi": {"seguridad": 3, "gobernanza": 3, "operabilidad": 4},
        "cloudformation": {"seguridad": 1, "gobernanza": 1, "operabilidad": 1},
        "bicep": {"seguridad": 5, "gobernanza": 4, "operabilidad": 5},
    },
    "rehacer_infra_post_incidente": {
        "terraform": {"seguridad": 5, "gobernanza": 5, "operabilidad": 4},
        "pulumi": {"seguridad": 4, "gobernanza": 4, "operabilidad": 4},
        "cloudformation": {"seguridad": 3, "gobernanza": 4, "operabilidad": 3},
        "bicep": {"seguridad": 3, "gobernanza": 4, "operabilidad": 3},
    },
}

def score_total(metricas: dict[str, int]) -> float:
    return round(sum(metricas[k] * pesos[k] for k in pesos), 2)

def recomendacion(opciones: dict[str, dict[str, int]]) -> tuple[str, float]:
    ranking = {tool: score_total(vals) for tool, vals in opciones.items()}
    mejor = max(ranking, key=ranking.get)
    return mejor, ranking[mejor]

print("Sugerencia inicial ponderada por endurecimiento")
print("=" * 72)
for nombre, opciones in escenarios.items():
    herramienta, puntaje = recomendacion(opciones)
    print(f"\nEscenario: {nombre}")
    print(f"  Herramienta sugerida: {herramienta} (score={puntaje})")
    print("  Verificar además: evidencia de aprobación, plan firmado y pruebas post-cambio.")

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>